<a href="https://colab.research.google.com/github/hieuvu-0111/Deep_Learning_RNN/blob/main/Deep_Learning_RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Recurrent Neural Network (RNN) {-}

This notebook is adapted from a Deep Learning course assignment which is designed to help students understand how recurrent neural networks are applied to a text classification task using the IMDB movie review dataset. We will go through the complete workflow, including loading the dataset, analyzing and preprocessing text data, building different RNN based models, and evaluating their performance. By experimenting with GRU, LSTM, and bidirectional variants, we will gain practical insight into how different RNN architectures affect classification accuracy and runtime efficiency. The tasks include:

1.  **Coding tasks:**

    1.1 Analyze and preprocess the dataset to make it suitable for training RNN based models.  

    1.2 Build an RNN model using GRU (Gated Recurrent Unit, https://www.tensorflow.org/api_docs/python/tf/keras/layers/GRU) instead of LSTM, following the demo code provided. Train the model and evaluate its performance on the test set.  

    1.3 Build an RNN model using Bidirectional LSTM (BiLSTM, https://www.tensorflow.org/api_docs/python/tf/keras/layers/Bidirectional). Train the model and evaluate its performance on the test set.  

    1.4 Build an RNN model using Bidirectional GRU (BiGRU, https://www.tensorflow.org/api_docs/python/tf/keras/layers/Bidirectional). Train the model and evaluate its performance on the test set.  

    1.5 Compare the models built in this assignment, including LSTM, GRU, BiLSTM, and BiGRU, in terms of classification accuracy and runtime efficiency. Discuss your observations and explain the performance differences among these model variants.  

2.  **Open discussion questions:**

    2.1 During preprocessing, text is converted into sequences of integers. Discuss how the choice of vocabulary size and handling of rare words can affect both model performance and generalization on unseen reviews.  
    2.2 Bidirectional RNNs process sequences in both forward and backward directions. Discuss how this bidirectional context can improve sentiment classification and provide examples of review patterns where this might be especially helpful.  
    2.3 Is the IMDB dataset class imbalanced, and how does its class distribution influence the selection of evaluation metrics? In your opinion, which evaluation metrics should be used to better understand the model’s performance, and why?  
    2.4 Sequence length truncation is often applied to limit computational cost. Discuss the trade off between keeping longer sequences for richer context and truncating them for faster training and lower memory usage.  
    2.5 Based on your overall findings, propose improvements or extensions to this assignment. These may include architectural changes, different preprocessing strategies, or alternative evaluation methods, and explain how they could lead to better sentiment classification performance.  


The dataset we will be working on is imdb_reviews. This dataset is a large movie review dataset. This dataset is for binary sentiment classification containing a set of 25,000 highly polar movie reviews for training, and 25,000 for testing. All the reviews have either a positive or negative sentiment. Reference: http://ai.stanford.edu/~amaas/data/sentiment/

Each data sample contains:
- label (tf.int64)
- text (tf.string)

In [ ]:
# Note: to enable GPU training in Colab, go to Runtime > Change runtime type > Hardware acceleration > Choose GPU from the drop-down list.

# Download tensorflow datasets
!pip install tensorflow_datasets

# Import libraries
import numpy as np
import tensorflow_datasets as tfds
import tensorflow as tf
import matplotlib.pyplot as plt

In [ ]:
# Load the IMDB movie review dataset, return text (movie review) and label (positive/negative)
train_dataset, val_dataset, test_dataset = tfds.load(name="imdb_reviews", split=('train[:80%]', 'train[80%:]', 'test'), as_supervised=True)

print("Training set: ", len(train_dataset), "samples")
print("Validation set: ", len(val_dataset), "samples")
print("Test set: ", len(test_dataset), "samples")

In [ ]:
# Show same samples in the training set
for example, label in train_dataset.take(3):
  print('text: ', example.numpy())
  print('label: ', label.numpy())

## 1. Coding tasks

### 1.1 (1 point) Analyze and preprocess the dataset to make it suitable for training RNN based models.


In [ ]:
### Convert Tensorflow Dataset to numpy arrays of feature vector X and label y

# Convert training set
train_ds_numpy = tfds.as_numpy(train_dataset) # Convert TF Dataset to an iterable of numpy array
train_numpy = np.vstack(list(train_ds_numpy)) # Stack to full numpy array
X_train = tf.convert_to_tensor(list(map(lambda x: x[0], train_numpy)), dtype=tf.string) # Extract review (index 0) from numpy vector
y_train = np.array(list(map(lambda x: x[1], train_numpy))).astype(np.int16) # Extract label (index 1) from numpy vector and convert grom string to number
print("X_train shape: " + str(X_train.shape))
print("y_train shape: " + str(y_train.shape))

# Convert validation set
val_ds_numpy = tfds.as_numpy(val_dataset) # Convert TF Dataset to an iterable of numpy array
val_numpy = np.vstack(list(val_ds_numpy)) # Stack to full numpy array
X_val = tf.convert_to_tensor(list(map(lambda x: x[0], val_numpy)), dtype=tf.string) # Extract review (index 0) from numpy vector
y_val = np.array(list(map(lambda x: x[1], val_numpy))).astype(np.int16) # Extract label (index 1) from numpy vector
print("X_val shape: " + str(X_val.shape))
print("y_val shape: " + str(y_val.shape))

# Convert test set
test_ds_numpy = tfds.as_numpy(test_dataset) # Convert TF Dataset to an iterable of numpy array
test_numpy = np.vstack(list(test_ds_numpy)) # Stack to full numpy array
X_test = tf.convert_to_tensor(list(map(lambda x: x[0], test_numpy)), dtype=tf.string) # Extract review (index 0) from numpy vector
y_test = np.array(list(map(lambda x: x[1], test_numpy))).astype(np.int16) # Extract label (index 1) from numpy vector
print("X_test shape: " + str(X_test.shape))
print("y_test shape: " + str(y_test.shape))

In [ ]:
### TextVectorization layer maps text features to integer sequences.
# Set vocabulary size for the training data
VOCAB_SIZE = 1000

# Initialize the TextVectorization layer for raw text encoding
text_encoder = tf.keras.layers.TextVectorization(max_tokens=VOCAB_SIZE) # Maximum size of the vocabulary

# Feed training text to adapt() method to calculate the layer's vocabulary
text_encoder.adapt(X_train)

In [ ]:
# Show the first 20 tokens. After the padding ('') and unknown ([UNK]) tokens they're sorted by frequency.
vocab = np.array(text_encoder.get_vocabulary()) # Get the vocabulary of the training set after the adaptation
print("Vocabulary size:", vocab.shape)
vocab[:20] # Show the first 20 tokens (sorted by frequency) in the vocabulary

In [ ]:
### Example of how a text vectorization layer works. It maps strings to integers.

# Create the model that uses the text vectorization layer
model_encoder = tf.keras.models.Sequential()

# Creating an explicit input layer.
# It needs to have a shape of (1,) (because we need to guarantee that there is exactly one string input per batch),
model_encoder.add(tf.keras.Input(shape=(1,), dtype=tf.string))

# The first (unique) layer in the model is the vectorization layer.
# After this layer, we have a tensor of shape (batch_size, max_len) containing vocab indices.
model_encoder.add(text_encoder)

# The model can map strings to integers, and you can add an embedding layer to map these integers to learned embeddings.
test_data = [["I want to drink"], ["I do not want to eat but I want to sleep"]]
test_data = tf.convert_to_tensor(test_data, dtype=tf.string)
model_encoder.predict(test_data)

# Note: The tensors of indices are zero-padded to the longest sequence in the batch,
# to make sure that all tensors of indices have the same size (for batching purpose).

In [ ]:
# Note: the vectorization process is not completely reversible due to the limited vocabulary size
print("Original: ", X_train[0])
print()
print("Round-trip: ", " ".join(vocab[text_encoder(X_train[0]).numpy()]))

### 1.2 (1 point) Build an RNN model using GRU (Gated Recurrent Unit, https://www.tensorflow.org/api_docs/python/tf/keras/layers/GRU) instead of LSTM, following the demo code provided. Train the model and evaluate its performance on the test set.


In [ ]:
# Initialize a sequential model since all the layers in the model only have single input and produce single output
model = tf.keras.Sequential([
    tf.keras.Input(shape=(1,), dtype=tf.string),
    text_encoder, # Text encoder layer, i.e., TextVectorization layer
    tf.keras.layers.Embedding( # Text embedding layer, i.e., turns positive integers (indexes) into dense vectors of fixed size.
        input_dim=len(text_encoder.get_vocabulary()), # Get the size of word vocabulary (positive integers), VOCAB_SIZE.
        output_dim=32, # Fixed size of the output embedding vectors which is fed as input of GRU layer.
        mask_zero=True), # Whether or not the input value 0 (the zero-padding) should be masked out
        # Hence if mask_zero = True, index 0 cannot be used in the vocabulary (input_dim = vocabulary size + 1)
    tf.keras.layers.GRU(units=32, dropout=0.2, kernel_regularizer=tf.keras.regularizers.L2(0.01)), # Output dimension of GRU layer
    tf.keras.layers.Dense(32, activation='relu', kernel_regularizer=tf.keras.regularizers.L2(0.01)), # Dense layer
    tf.keras.layers.Dense(1, activation='sigmoid') # Classification output
])

# Summarize the model
model.summary()

# The embedding layer uses masking to handle the varying sequence-lengths
# Check if the layers after the Embedding supports masking
print([layer.supports_masking for layer in model.layers])

In [ ]:
# Compile the model
model.compile(loss='binary_crossentropy', # Binary classification loss
              optimizer=tf.keras.optimizers.Adam(1e-3), # Optimizer
              metrics=['accuracy']) # Evaluation metric

checkpoint_gru = tf.keras.callbacks.ModelCheckpoint(
    "gru_best.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=5, # Stop training if val_accuracy doesn't improve for 5 epochs
    restore_best_weights=True,
    verbose=1
)

# Train the model
history = model.fit(X_train, y_train, epochs=20, batch_size = 128, validation_data=(X_val, y_val), callbacks=[checkpoint_gru, early_stopping])

In [ ]:
# Visualize the training and validation loss over epochs
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'val'], loc='upper right')
plt.show()

In [ ]:
# Visualize the training and validation accuracy over epochs
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'val'], loc='upper left')
plt.show()

In [ ]:
# Evaluate the model on the test set
test_loss, test_acc = model.evaluate(X_test, y_test)
print('Test Loss:', test_loss)
print('Test Accuracy:', test_acc)

In [ ]:
# Make prediction on a new data sample
sample_reviews = [('It is a cool movie. The graphics and the animation are awesome.'),
                  ('The movie was really bad. I would not recommend it to anyone.')]
sample_reviews = tf.convert_to_tensor(sample_reviews, dtype=tf.string)
predictions = model.predict(sample_reviews)
print(predictions[0])
print(predictions[1])

### 1.3 (1 point) Build an RNN model using Bidirectional LSTM (BiLSTM, https://www.tensorflow.org/api_docs/python/tf/keras/layers/Bidirectional). Train the model and evaluate its performance on the test set.


In [ ]:
# Initialize a sequential model for BiLSTM
model_bilstm = tf.keras.Sequential([
    tf.keras.Input(shape=(1,), dtype=tf.string),
    text_encoder,
    tf.keras.layers.Embedding(
        input_dim=len(text_encoder.get_vocabulary()),
        output_dim=32,
        mask_zero=True),

    tf.keras.layers.Bidirectional(
        tf.keras.layers.LSTM(units=32, dropout=0.2, kernel_regularizer=tf.keras.regularizers.L2(0.01))
  ),
    tf.keras.layers.Dense(32, activation='relu', kernel_regularizer=tf.keras.regularizers.L2(0.01)),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# Summarize the model
model_bilstm.summary()

# Check if the layers after the Embedding supports masking
print([layer.supports_masking for layer in model_bilstm.layers])

In [ ]:
# Compile the BiLSTM model
model_bilstm.compile(loss='binary_crossentropy',
                     optimizer=tf.keras.optimizers.Adam(1e-3),
                     metrics=['accuracy'])

checkpoint_bilstm = tf.keras.callbacks.ModelCheckpoint(
    "bilstm_best.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

early_stopping_bilstm = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=5, # Stop training if val_accuracy doesn't improve for 5 epochs
    restore_best_weights=True,
    verbose=1
)

# Train the BiLSTM model
history_bilstm = model_bilstm.fit(X_train, y_train,
                                  epochs=20, batch_size=128,
                                  validation_data=(X_val, y_val), callbacks=[checkpoint_bilstm, early_stopping_bilstm])

In [ ]:
# Visualize the training and validation loss over epochs
plt.plot(history_bilstm.history['loss'])
plt.plot(history_bilstm.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'val'], loc='upper right')
plt.show()

In [ ]:
# Visualize the training and validation accuracy over epochs
plt.plot(history_bilstm.history['accuracy'])
plt.plot(history_bilstm.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'val'], loc='upper left')
plt.show()

In [ ]:
# Evaluate the model on the test set
test_loss, test_acc = model_bilstm.evaluate(X_test, y_test)
print('Test Loss:', test_loss)
print('Test Accuracy:', test_acc)

In [ ]:
# Make prediction on a new data sample
sample_reviews = [('It is a cool movie. The graphics and the animation are awesome.'),
                  ('The movie was really bad. I would not recommend it to anyone.')]
sample_reviews = tf.convert_to_tensor(sample_reviews, dtype=tf.string)
predictions = model_bilstm.predict(sample_reviews)
print(predictions[0])
print(predictions[1])

### 1.4 (1 point) Build an RNN model using Bidirectional GRU (BiGRU, https://www.tensorflow.org/api_docs/python/tf/keras/layers/Bidirectional). Train the model and evaluate its performance on the test set.


In [ ]:
# Initialize a sequential model for BiGRU
model_bigru = tf.keras.Sequential([
    tf.keras.Input(shape=(1,), dtype=tf.string),
    text_encoder,
    tf.keras.layers.Embedding(
        input_dim=len(text_encoder.get_vocabulary()),
        output_dim=32,
        mask_zero=True),
    tf.keras.layers.Bidirectional(
        tf.keras.layers.GRU(units=32, dropout=0.2, kernel_regularizer=tf.keras.regularizers.L2(0.01))
  ),
    tf.keras.layers.Dense(32, activation='relu', kernel_regularizer=tf.keras.regularizers.L2(0.01)),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# Summarize the model
model_bigru.summary()

# Check if the layers after the Embedding supports masking
print([layer.supports_masking for layer in model_bigru.layers])

In [ ]:
# Compile the BiGRU model
model_bigru.compile(loss='binary_crossentropy',
                      optimizer=tf.keras.optimizers.Adam(1e-3),
                      metrics=['accuracy'])

checkpoint_bigru = tf.keras.callbacks.ModelCheckpoint(
    "bigru_best.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)


early_stopping_bigru = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=5, # Stop training if val_accuracy doesn't improve for 5 epochs
    restore_best_weights=True,
    verbose=1
)

# Train the BiGRU model
history_bigru = model_bigru.fit(X_train, y_train,
                                  epochs=20, batch_size=128,
                                  validation_data=(X_val, y_val), callbacks=[checkpoint_bigru, early_stopping_bigru])

In [ ]:
# Visualize the training and validation loss over epochs
plt.plot(history_bigru.history['loss'])
plt.plot(history_bigru.history['val_loss'])
plt.title('BiGRU Model Loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'val'], loc='upper right')
plt.show()

In [ ]:
# Visualize the training and validation accuracy over epochs
plt.plot(history_bigru.history['accuracy'])
plt.plot(history_bigru.history['val_accuracy'])
plt.title('BiGRU Model Accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'val'], loc='upper left')
plt.show()

In [ ]:
# Evaluate the BiGRU model on the test set
test_loss_bigru, test_acc_bigru = model_bigru.evaluate(X_test, y_test)
print('BiGRU Test Loss:', test_loss_bigru)
print('BiGRU Test Accuracy:', test_acc_bigru)

In [ ]:
# Make prediction on a new data sample
sample_reviews_bigru = [('It is a fantastic movie. The storytelling and acting are superb.'),
                        ('This film was incredibly boring and a waste of time.')]
sample_reviews_bigru = tf.convert_to_tensor(sample_reviews_bigru, dtype=tf.string)
predictions_bigru = model_bigru.predict(sample_reviews_bigru)
print(predictions_bigru[0])
print(predictions_bigru[1])

### 1.5 (1 point) Compare the models built in this assignment, including LSTM, GRU, BiLSTM, and BiGRU, in terms of classification accuracy and runtime efficiency. Discuss your observations and explain the performance differences among these model variants.

**In terms of classification accuracy on the test set:**
*   GRU Model: 86.10%
*   BiLSTM Model: 86.05%
*   BiGRU Model: 86.21%

We see that all three models achieved very similar test accuracies (around 86%). However, if we need to compare, the highest value belongs to the BiGRU model, followed closely by the standard GRU and then BiLSTM. This indicates that for this specific task, with the current setup (for example, vocabulary size, batch size), all these recurrent architectures are capable of capturing sentiment with comparable effectiveness.

**In terms of the running time:**

*   GRU Model:
    *   Training: Approximately 7-10 seconds per epoch.
    *   Evaluation: ~11 seconds.
    *   Model stops running at epoch: 19

*   BiLSTM Model:
    *   Training: Approximately 12-21 seconds per epoch.
    *   Evaluation: ~18 seconds.
    *   Model stops running at epoch: 8

*   BiGRU Model:
    *   Training: Approximately 12-15 seconds per epoch.
    *   Evaluation: ~18 seconds.
    *   Model stops running at epoch: 10

**In terms of numbers of parameters:**
*   GRU model: 39,425
*   BiLSTM model: 50,753
*   BiGRU model: 46,785

**Discussion**
We expect that with bidirectional approach, the test accuracy would be higher than that of the original GRU, but in fact there is a very small difference in test accuracy across all three models. This suggests that for this particular dataset and model configuration, the additional complexity of bidirectional processing or the architectural differences between GRU and LSTM gates do not lead into significant accuracy rise (the simple GRU performs almost as well as its bidirectional and LSTM models.)

About the running time, both BiLSTM and BiGRU models exhibited noticeably longer training times per epoch and higher evaluation times compared to the unidirectional GRU model.  Because bidirectional layers effectively run two recurrent networks (forward and backward), the (almost) doubled running time is expected.

Although having pretty similar accuracy, bidirectional RNNs have more numbers of parameters. The highest value belongs to BiLSTM, then BiGRU, and GRU has the lowest value.

Although not clearly reflected in significantly higher accuracy in this specific experiment, Bidirectional RNNs are theoretically advantageous for tasks like sentiment classification. They allow the model to capture context from both sides, which may be crucial for understanding nuanced sentiment, especially in sentences where the sentiment might be inverted by a later word. We will discuss more about this phenomenon on question 2.2.

## 2. Open discussion questions

### 2.1 (1 point) During preprocessing, text is converted into sequences of integers. Discuss how the choice of vocabulary size and handling of rare words can affect both model performance and generalization on unseen reviews.

In terms of choosing vocabulary size, we consider the pros and cons of choosing a smaller size and bigger size.

A smaller vocabulary size, like 1000 words set in our model, may reduce model complexity and memory needed, which can result in faster training due to fewer parameters in the embedding layer. It also forces the model to rely on more frequent words to capture more general and broader patterns in sequences. However, with a limited size, many unique words will be mapped as unknown. This may lead to information loss, especially for important words not appearing frequently. Then the model might struggle to understand and classify that case. Generally, the model can be underfitted if too many important words are ignored. It might perform well on test data if the sentiment is primarily determined by common words, but it will generalize poorly to reviews containing a high proportion of out-of-vocabulary words or where sentiment depends on specific rare ones.

On the other hand, a larger vocabulary size, e.g., 50,000 words, can capture a wider range of words, including more specific and nuanced terms. Thus, it can has a richer understanding of sentiment. However, increaing vocab size means increasing model complexity, memory usage (larger embedding layer), and training time. Then. more parameters mean a higher risk of overfitting if the dataset is not large or diverse enough. In general, the model can potentially achieve higher accuracy, but it requires more data to train effectively. If the vocabulary is too large relative to the training data, the model might overfit to specific word occurrences in the training set, thus leading to poor generalization on unseen reviews that use these words in different contexts or simply do not contain them.


For handling rare words, we can map them to unknown tokens or exclude them entirely.

Mapping to unknown token is the most common approach, and it is what `TextVectorization` does by default beyond the `max_tokens` limit. All words not in the predefined vocabulary are mapped to a single `[UNK]` token. In so doing, it simplifies the vocabulary but loses the specific meaning of each rare word. If rare words carry significant sentiment, this can harm the performance. At the same time, it helps generalization by preventing the model from overfitting to specific rare words. However, it can also hinder generalization if crucial rare words in unseen data are simply treated as unknown tokens, thus reducing model accuracy.

If rare words are simply dropped, information is lost, which is similar to mapping to `[UNK]` but ensures the model does not even see the presence of such tokens.

A more advanced technique is subword tokenization, e.g., WordPiece, BPE. This technique breaks down rare words into common subword units. For example, 'unbelievable' might become 'un', 'believe', 'able'. Then, it reduces the number of `[UNK]` tokens significantly, as even rare words can be represented by combinations of known subwords. Thus, it retains more semantic information than simply mapping to `[UNK]`. Subword tokenization can improve generalization, especially for morphologically rich languages or when dealing with unseen words. The model can infer the meaning of new words from its known subword components.



### 2.2 (1 point) Bidirectional RNNs process sequences in both forward and backward directions. Discuss how this bidirectional context can improve sentiment classification and provide examples of review patterns where this might be especially helpful.


There are three following cases when bi-RNNs prove its better performance compared to uni-RNN.

1.  First, it can better handle the late negation in a review. A unidirectional RNN might get stuck on the positive words before it even reaches the negative conclusion. Meanwhile, bi-RNN, by explore both directions of the sentence, can understand more clearly the context and capture the broader patterns. For example, for a review "This film is good, but I do not like the ending at all". When uni-RNN read the word "good", it built a strong positive sentiment. By the time it hits "do not like," the positive momentum might be too high to fully flip the sentiment to negative. In bi-RNN, the backward pass saw the negative words and told the model to reassess the sentiment, leading to more accurate result.


2.  Second, it helps better identify sarcasm and irony. Again, in this case, with two passes, Bi-RNN build a broader and clearer context to understand the true meaning of the whole sentence. For instance, a review of a thriller is "This movie is a masterpiece of how to absolutely not make a thriller". Uni-RNN sees "masterpiece" and immediately labels the review as highly positive. It does not realize the sarcasm until the very last word. For Bi-RNN, because it processes the sentence from both ends, the model sees "not make the thriller" and "masterpiece" at the same time, these two contexts complement each other, allowing the model to correctly identify the sarcastic and negative tone of the review.

3.  Third, it enhance the model understanding of conceptual feature in a reviwq. In a review, a specific movie element (like "the plot" or "the lead actor") is neutral until the descriptive adjective appears. A Bi-RNN helps the model "anchor" the sentiment to the correct feature. For example, "the pacing, though slow at first, eventually became gripping". When uni-RNN reaches the word "pacing", it only knows the "slow" part (negative/neutral) but it hasn't seen the "gripping" part yet.Meanwhile, the Bi-RNN has information from both sides. It knows "pacing" is the subject (forward) and it knows "gripping" is the final verdict (backward). This allows the model to correctly weigh the review as positive overall.


### 2.3 (1 point) Is the IMDB dataset class imbalanced, and how does its class distribution influence the selection of evaluation metrics? In your opinion, which evaluation metrics should be used to better understand the model’s performance, and why?


In [ ]:
# Check label distribution in dataset

unique, counts = np.unique(y_train, return_counts=True)
label_distribution = dict(zip(unique, counts))

print("Label distribution in training set:")
for label, count in label_distribution.items():
    print(f"  Label {label}: {count} samples")

if len(unique) == 2:
    ratio = counts[0] / counts[1]
    print(f"Ratio of class 0 to class 1: {ratio:.2f}")

unique_val, counts_val = np.unique(y_val, return_counts=True)
print("\nLabel distribution in validation set:", dict(zip(unique_val, counts_val)))
unique_test, counts_test = np.unique(y_test, return_counts=True)
print("\nLabel distribution in test set:", dict(zip(unique_test, counts_test)))

As we may see from the above code result, the IMDB dataset is not imbalanced. In all training, validation and test set, the ratio of class 0 to class 1 is approximately 1.

For a specific dataset, the class distribution has a significant influence on the reliability of certain evaluation metrics:

*   Accuracy: When classes are balanced, accuracy is a reliable and informative metric. It directly reflects the proportion of correctly classified instances out of the total. If the dataset were imbalanced (e.g., 90% positive, 10% negative), a model that simply predicted 'positive' for all instances could achieve 90% accuracy, making accuracy misleading. However, with a 50/50 split, a high accuracy genuinely indicates good performance across both classes.

*   Precision, Recall, F1-Score: While accuracy is good for balanced datasets, metrics like Precision, Recall, and espacially F1-Score become important for imbalanced datasets as they provide a more nuanced view of performance for each class. It tells us how much the model cover the examined label among all label or how much labels are predicted correctly among all predicted labels. Also, or balanced datasets, they still offer valuable insights into specific types of errors (false positives vs. false negatives) even if accuracy looks good.

For a balanced dataset like IMDB, while accuracy is a good starting point, we may use other metrics to gain more comprehensive understanding on model performance, for example, precision, recall, f1-score and confusion matrix. Let's analyze the meaning of each metric.

1.  Precision: it measures the proportion of true positive predictions among all positive predictions made by the model. In sentiment analysis, high precision for 'positive' means that when the model says a review is positive, it's usually correct. Conversely, high precision for 'negative' means when the model says a review is negative, it's usually correct. This helps understand the cost of false positives (e.g., classifying a negative review as positive).

2.  Recall: it measures the proportion of true positive predictions among all actual positive instances. High recall for 'positive' means the model correctly identifies most of the actual positive reviews. High recall for 'negative' means it catches most actual negative reviews. This helps understand the cost of false negatives (e.g., failing to identify an actual negative review).

3.  F1-Score: it is the harmonic mean of precision and recall. It provides a single metric that balances both precision and recall. It's especially useful when you want to ensure a good balance between identifying relevant instances (recall) and not classifying irrelevant instances as relevant (precision).

4.  Confusion Matrix: This is a table providing a complete breakdown of correct and incorrect classifications for each class. It shows True Positives (TP), True Negatives (TN), False Positives (FP), and False Negatives (FN). It is valuable for visualizing the types of errors the model is making and for calculating precision, recall, and F1-score.

### 2.4 (1 point) Sequence length truncation is often applied to limit computational cost. Discuss the trade off between keeping longer sequences for richer context and truncating them for faster training and lower memory usage.


Sequence length truncation is a common preprocessing step in NLP and RNN. It sets a maximum length for input sequences, then if a sequence is longer than this maximum, it's cut off; if it's shorter, it's often padded to reach the maximum length.

On the one hand, compared to truncating sequences, keeping the original has some positive effect. Indeed, longer sequences inherently contain more words and thus more context. For tasks like sentiment analysis, crucial information (e.g., nuanced phrasing, specific details, overall tone) might be spread across many words. Keeping longer sequences allows the model to potentially capture all this relevant information. Especially, in complex sentences, the original sentence will provide deeper understanding of linguistic nuances, sarcasm, or complex logical structures. Also, for tasks where long-range dependencies are vital such as detailed question answering, summarizing documents, or understanding reviews, longer sequences generally lead to better performance.

On the other hand, there are certain advantages of truncation over keeping the original sequences. Obviously, processing longer sequences requires more computational resources. This means longer training times for each epoch and slower inference. Along with longer time is the larger memory. Long sequences need more memory, which can quickly lead to out-of-memory errors during training process. Also, while LSTMs and GRUs are designed to mitigate vanishing and exloiding gradient issues, extremely long sequences can still make the problem come back, thus it it harder for the model to learn effective long-range dependencies.

We can summarize the pros and cons of two approach as follows:

| Feature                    | Longer Sequences                                      | Truncated Sequences                                   |
| :------------------------- | :---------------------------------------------------- | :---------------------------------------------------- |
| **Context Captured**       | Richer, more complete                                 | Limited, potentially losing critical information      |
| **Computational Cost**     | High (slower training/inference)                      | Low (faster training/inference)                       |
| **Memory Usage**           | High (risk of OOM errors)                             | Low (allows larger batch sizes)                       |
| **Learning Dependencies**  | Better for long-range dependencies (with good RNNs)   | Might struggle if key info is truncated               |
| **Risk of Problems**       | Vanishing/exploding gradients (for very long)         | Information loss, suboptimal performance              |



### 2.5 (1 point) Based on your overall findings, propose improvements or extensions to this assignment. These may include architectural changes, different preprocessing strategies, or alternative evaluation methods, and explain how they could lead to better sentiment classification performance.

Throughout implementing the model and explore topics on discussion questions, we obtain some following insights:

*   Make use of pre-trained word embeddings: Instead of training embeddings from scratch as we did, we can leverage pre-trained embeddings like Word2Vec, GloVe, or fastText Embeddings are designed to capture semantic relationships between words. Words with similar meanings (e.g., "good" and "great") will have similar embedding vectors, while dissimilar words will have distant vectors. This allows the model to generalize better by understanding the nuances of language. Also, it reduces the dimensionality of features, making training more efficient and optimal.

*   Add Recurrent Dropout: Our current models utilized standard `dropout` but did not implement `recurrent_dropout` within the GRU/LSTM layers, partly to manage training tim, after realizing that training with recurrent dropout takes too much time and memory (about 30 minutes/epoch in my computer). Despite prolonging the training process, it is specifically designed to regularize the recurrent connection to overfitting more effectively in RNNs. By applying dropout consistently across time steps to the hidden state, it can significantly improve the model's generalization capabilities. Then we can expect the model to have more robust performance on the test set.

*   Apply advanced preprocessing techniques, such as subword tokenization, the technique breaks words into smaller and meaningful units to effectively handle unknown words by representing them as combinations of known subwords. This significantly reduces information loss due to `[UNK]` tokens, thus improving model performance.

*   Hybrid CNN and RNN Models: Combining CNN and RNN is expected to significantly enhance review classification, especially in complex sentences or longer reviews. CNNs robust feature extraction and RNNs enhance sequential context understanding. Indeed, CNNs are excellent at extracting local features and n-gram patterns from text, effectively identifying important phrases or keywords that indicate sentiment, regardless of their position in the sequence. For example, a CNN layer can detect patterns like "not good at all" or "highly recommended". When these features are then fed into an RNN (like LSTM or GRU), the RNN can process these extracted local patterns sequentially, thereby capturing long-range dependencies and the overall contextual flow of the review.